In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import utils_z
import cityjson_parser as cjpar
import time

In [6]:
conn = utils_z.get_conn("Test20260413", "postgres", "we6666", "localhost", "5432")

### 1. 处理 CityGML 数据并导入数据库

#### 1.1 测试：使用 citygml-tools 将1个 lod1 CityGML 转换为 CityJSON

In [8]:
citygml_tools_path = r"E:\0_mylib\citygml-tools-2.4.1\citygml-tools.bat"

utils_z.run_cmd(f'"{citygml_tools_path}" --version')

citygml-tools version 2.4.1
(C) 2018-2026 Claus Nagel <claus.nagel@gmail.com>




CompletedProcess(args='"E:\\0_mylib\\citygml-tools-2.4.1\\citygml-tools.bat" --version', returncode=0, stdout='citygml-tools version 2.4.1\n(C) 2018-2026 Claus Nagel <claus.nagel@gmail.com>\n\n', stderr='')

In [9]:
test_xml_path = "E:\\2_data\\building_3d_opensource\\hamburg\\LoD2-DE_HH_2025-03-14\\LoD2_32_557_5932_1_HH.xml"

utils_z.run_cmd(f'"{citygml_tools_path}" to-cityjson {test_xml_path}')


[10:27:39 INFO] Starting citygml-tools.
[10:27:39 INFO] Executing 'to-cityjson' command.
[10:27:40 INFO] [1|1] Processing file E:\2_data\building_3d_opensource\hamburg\LoD2-DE_HH_2025-03-14\LoD2_32_557_5932_1_HH.xml.
[10:27:40 INFO] Writing output to file E:\2_data\building_3d_opensource\hamburg\LoD2-DE_HH_2025-03-14\LoD2_32_557_5932_1_HH.json.
[10:27:41 INFO] Total execution time: 01 s.
[10:27:41 INFO] citygml-tools successfully completed.



CompletedProcess(args='"E:\\0_mylib\\citygml-tools-2.4.1\\citygml-tools.bat" to-cityjson E:\\2_data\\building_3d_opensource\\hamburg\\LoD2-DE_HH_2025-03-14\\LoD2_32_557_5932_1_HH.xml', returncode=0, stdout="[10:27:39 INFO] Starting citygml-tools.\n[10:27:39 INFO] Executing 'to-cityjson' command.\n[10:27:40 INFO] [1|1] Processing file E:\\2_data\\building_3d_opensource\\hamburg\\LoD2-DE_HH_2025-03-14\\LoD2_32_557_5932_1_HH.xml.\n[10:27:40 INFO] Writing output to file E:\\2_data\\building_3d_opensource\\hamburg\\LoD2-DE_HH_2025-03-14\\LoD2_32_557_5932_1_HH.json.\n[10:27:41 INFO] Total execution time: 01 s.\n[10:27:41 INFO] citygml-tools successfully completed.\n", stderr='')

#### 1.2 测试：解析并检查一个 CityJSON 数据

In [10]:
import json

test_json_path = test_xml_path.replace('.xml', '.json')
with open(test_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 查看顶层结构
print(data.keys())

# 查看第一栋建筑的属性
first_building_id = list(data["CityObjects"].keys())[0]
first_building = data["CityObjects"][first_building_id]
print(f"\n建筑ID：{first_building_id}")
print(f"类型：{first_building['type']}")
print(f"属性：{json.dumps(first_building.get('attributes', {}), indent=2, ensure_ascii=False)}")
print(f"几何类型：{[g['type'] for g in first_building.get('geometry', [])]}")

dict_keys(['type', 'version', 'CityObjects', 'transform', 'vertices', 'metadata'])

建筑ID：DEHHALKAJ0000nVM
类型：Building
属性：{
  "creationDate": "2023-02-23T00:00:00+08:00",
  "DatenquelleLage": "1000",
  "DatenquelleBodenhoehe": "1100",
  "Gemeindeschluessel": "02100141",
  "DatenquelleDachhoehe": "1000",
  "DatenquelleGeschossanzahl": "1000",
  "Geometrietyp2DReferenz": "3000",
  "Grundrissaktualitaet": "2025-01-27",
  "measuredHeight": 2.297,
  "function": "31001_2460",
  "roofType": "1000",
  "storeysAboveGround": 1
}
几何类型：['Solid']


In [11]:
first_geom = first_building["geometry"][0]
print(f"几何类型：{first_geom['type']}")
print(f"lod：{first_geom.get('lod')}")

# 看看boundaries结构
boundaries = first_geom["boundaries"]
print(f"faces数量：{len(boundaries[0])}")
print(f"第一个face：{boundaries[0][0]}")

# 看看transform
print(f"\ntransform：{json.dumps(data['transform'], indent=2)}")
print(f"vertices总数：{len(data['vertices'])}")
print(f"前3个顶点：{data['vertices'][:3]}")

几何类型：Solid
lod：2
faces数量：6
第一个face：[[0, 1, 2, 3]]

transform：{
  "scale": [
    0.001,
    0.001,
    0.001
  ],
  "translate": [
    556975.486,
    5931992.369,
    1.289
  ]
}
vertices总数：7113
前3个顶点：[[649337, 96570, 6982], [649355, 101950, 6982], [640848, 101981, 6982]]


In [9]:
buildings = cjpar.parse_cityjson_lod1(test_json_path)

print(f"解析出建筑数：{len(buildings)}")
b = buildings[0]
print(f"ID：{b['building_id']}")
print(f"高度：{b['height']}")
print(f"楼层：{b['floor_count']}")
print(f"功能：{b['function']}")
print(f"2D footprint：{b['geom_2d']}")

解析出建筑数：1110
ID：DEHHALKA10005SAX
高度：6.88
楼层：2
功能：31001_1010
2D footprint：POLYGON ((549374.76 5936958.141, 549386.416 5936958.461, 549386.675 5936949.114, 549377.3200000001 5936948.865999999, 549375.024 5936948.796, 549374.888 5936953.3889999995, 549374.76 5936958.141))


#### 1.3 创建建筑lod1数据库表并批量转换、导入数据

In [ ]:
# xml批量转换为json
batch_json_output_dir = "E:\\2_data\\building_3d_opensource\\hamburg\\lod1_json"
batch_xml_input_dir = "E:\\2_data\\building_3d_opensource\\hamburg\\LoD1-DE_HH_2023-04-01"

xml_files = [f for f in os.listdir(batch_xml_input_dir) if f.endswith(".xml")]
print(f"共{len(xml_files)}个文件待转换")

errors = []
for i, filename in enumerate(xml_files):
    input_path = os.path.join(batch_xml_input_dir, filename)
    cmd = f'"{citygml_tools_path}" to-cityjson --output="{batch_json_output_dir}" "{input_path}"'
    try:
        utils_z.run_cmd(cmd)
        if (i + 1) % 50 == 0:
            print(f"进度：{i + 1}/{len(xml_files)}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"错误：{filename} -> {e}")

print(f"转换完成，失败{len(errors)}个")

In [3]:
# 创建LOD1建筑表
utils_z.run_sql("""
                CREATE TABLE IF NOT EXISTS hamburg_buildings_lod1
                (
                    building_id
                    VARCHAR
                    PRIMARY
                    KEY,
                    geom_2d
                    GEOMETRY
                (
                    Polygon,
                    25832
                ),
                    height FLOAT,
                    floor_count INTEGER,
                    function VARCHAR,
                    roof_type VARCHAR,
                    year_built INTEGER,
                    geom_3d GEOMETRY
                (
                    PolyhedralSurfaceZ,
                    25832
                )
                    );
                """, conn=conn)

utils_z.run_sql("CREATE INDEX IF NOT EXISTS buildings_lod1_geom_idx ON hamburg_buildings_lod1 USING GIST (geom_2d);",
                conn=conn)
print("LOD1建筑表创建完成")

LOD1建筑表创建完成


In [16]:
json_files = [f for f in os.listdir(batch_json_output_dir) if f.endswith(".json")]
print(f"共{len(json_files)}个文件待处理")

total = 0
errors = []

for i, filename in enumerate(json_files):
    filepath = os.path.join(batch_json_output_dir, filename)
    try:
        buildings = cjpar.parse_cityjson_lod1(filepath)
        print(f"\rProcessing {filename}: {len(buildings)} buildings", end='', flush=True)
        count = cjpar.insert_buildings_lod1(buildings, conn, lod1_table="hamburg_buildings_lod1")
        total += count
        if (i + 1) % 50 == 0:
            print()
            print(f"进度：{i + 1}/{len(json_files)}，已入库建筑：{total}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"错误：{filename} -> {e}")

print()  # To move to the next line after the loop
print(f"\n完成！共入库建筑：{total}")
print(f"失败文件数：{len(errors)}")

共784个文件待处理
Processing LoD1_32_554_5924_1_HH.json: 916 buildingss进度：50/784，已入库建筑：15328
Processing LoD1_32_557_5926_1_HH.json: 1033 buildings进度：100/784，已入库建筑：38867
Processing LoD1_32_559_5938_1_HH.json: 539 buildingss进度：150/784，已入库建筑：68189
Processing LoD1_32_561_5937_1_HH.json: 779 buildingss进度：200/784，已入库建筑：88981
Processing LoD1_32_563_5933_1_HH.json: 629 buildingss进度：250/784，已入库建筑：116545
Processing LoD1_32_565_5930_1_HH.json: 475 buildingss进度：300/784，已入库建筑：153498
Processing LoD1_32_567_5921_1_HH.json: 268 buildingss进度：350/784，已入库建筑：179724
Processing LoD1_32_568_5944_1_HH.json: 914 buildingss进度：400/784，已入库建筑：207030
Processing LoD1_32_570_5941_1_HH.json: 551 buildingss进度：450/784，已入库建筑：227768
Processing LoD1_32_572_5930_1_HH.json: 305 buildingss进度：500/784，已入库建筑：254366
Processing LoD1_32_573_5947_1_HH.json: 523 buildingss进度：550/784，已入库建筑：291165
Processing LoD1_32_575_5930_1_HH.json: 1 buildingssss进度：600/784，已入库建筑：314181
Processing LoD1_32_576_5945_1_HH.json: 632 buildingss进度：650/784，已入库建筑：

#### 1.4 更新block表

In [18]:
result = utils_z.run_sql("""
                         SELECT constraint_name
                         FROM information_schema.table_constraints
                         WHERE table_name = 'hamburg_blocks'
                           AND constraint_type = 'PRIMARY KEY';
                         """, conn=conn, fetch=True)

print(result)

[('blocks_pkey',)]


In [19]:
# 1. 添加新字段
utils_z.run_sql("""
                ALTER TABLE hamburg_blocks
                    ADD COLUMN IF NOT EXISTS city VARCHAR,
                    ADD COLUMN IF NOT EXISTS country VARCHAR,
                    ADD COLUMN IF NOT EXISTS area_m2 FLOAT;
                """, conn=conn)

# 2. 填入城市、国家、面积
utils_z.run_sql("""
                UPDATE hamburg_blocks
                SET city    = 'Hamburg',
                    country = 'Germany',
                    area_m2 = ST_Area(geom);
                """, conn=conn)

# 3. 删除主键约束
utils_z.run_sql("""
                ALTER TABLE hamburg_blocks
                DROP
                CONSTRAINT blocks_pkey;
                """, conn=conn)

# 4. 把block_id从INTEGER改为VARCHAR，同时替换为新格式
utils_z.run_sql("""
                ALTER TABLE hamburg_blocks
                ALTER
                COLUMN block_id TYPE VARCHAR
    USING 'DE_HH_' || LPAD(block_id::TEXT, 6, '0');
                """, conn=conn)

# 5. 重新设置主键
utils_z.run_sql("""
                ALTER TABLE hamburg_blocks
                    ADD PRIMARY KEY (block_id);
                """, conn=conn)

print("改造完成")

# 验证
result = utils_z.run_sql("""
                         SELECT block_id, city, country, area_m2
                         FROM hamburg_blocks LIMIT 5;
                         """, conn=conn, fetch=True)

for row in result:
    print(row)

改造完成
('DE_HH_000003', 'Hamburg', 'Germany', 146583.28380672264)
('DE_HH_000004', 'Hamburg', 'Germany', 68077.03149206015)
('DE_HH_000006', 'Hamburg', 'Germany', 42258.92792725274)
('DE_HH_000007', 'Hamburg', 'Germany', 64097.37637555693)
('DE_HH_000155', 'Hamburg', 'Germany', 18986.38463513086)


#### 1.5 街区与建筑空间叠合，记录建筑所属的街区

In [20]:
# 添加block_id字段
utils_z.run_sql("""
                ALTER TABLE hamburg_buildings_lod1
                    ADD COLUMN IF NOT EXISTS block_id VARCHAR;
                """, conn=conn)

# 建立空间索引
utils_z.run_sql("""
                CREATE INDEX IF NOT EXISTS buildings_lod1_geom_idx
                    ON hamburg_buildings_lod1 USING GIST (geom_2d);
                """, conn=conn)

print("字段和索引准备完成，开始空间叠合...")

字段和索引准备完成，开始空间叠合...


In [22]:
# 用重心落点规则批量判定
start_time = time.time()

utils_z.run_sql("""
                UPDATE hamburg_buildings_lod1 b
                SET block_id = (SELECT bl.block_id
                                FROM hamburg_blocks bl
                                WHERE ST_Within(ST_Centroid(b.geom_2d), bl.geom)
                    LIMIT 1
                    );
                """, conn=conn)

end_time = time.time()
elapsed_time = end_time - start_time
print(f"空间叠合完成，耗时：{elapsed_time:.2f}秒")


空间叠合完成，耗时：23.50秒


In [23]:
# 检查数据结果
result = utils_z.run_sql("""
                         SELECT COUNT(*)        AS total,
                                COUNT(block_id) AS matched,
                                COUNT(*)           FILTER (WHERE block_id IS NULL) AS unmatched
                         FROM hamburg_buildings_lod1;
                         """, conn=conn, fetch=True)

print(f"总建筑数：{result[0][0]}")
print(f"成功匹配block：{result[0][1]}")
print(f"未匹配block：{result[0][2]}")

总建筑数：384629
成功匹配block：354272
未匹配block：30357


#### 1.6 绘制检查

In [3]:
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
import py5

In [6]:
# 随机选取100个有建筑的block
sql_blocks = """
             SELECT block_id, geom
             FROM hamburg_blocks
             WHERE block_id IN (SELECT DISTINCT block_id
                                FROM hamburg_buildings_lod1
                                WHERE block_id IS NOT NULL)
             ORDER BY RANDOM() LIMIT 50; \
             """
gdf_blocks = gpd.read_postgis(sql_blocks, conn, geom_col="geom", crs="EPSG:25832")
blocks = list(gdf_blocks.itertuples(index=False, name='Block'))
print(f"随机选取了 {len(blocks)} 个block")

# 预加载所有建筑数据
buildings_data = {}
for block in blocks:
    block_id = block.block_id
    sql_buildings = f"""
        SELECT building_id, geom_2d
        FROM hamburg_buildings_lod1
        WHERE block_id = '{block_id}';
    """
    gdf_buildings = gpd.read_postgis(sql_buildings, conn, geom_col="geom_2d", crs="EPSG:25832")
    buildings_data[block_id] = list(gdf_buildings.geometry)

# 全局变量
current_index = 0


def all_polygons(g):
    if isinstance(g, Polygon):
        return [g]
    elif isinstance(g, MultiPolygon):
        return list(g.geoms)
    return []


def draw_polygon(poly, is_building=False):
    """绘制单个polygon，根据是否是建筑设置不同颜色"""
    coords = list(poly.exterior.coords)
    if is_building:
        py5.fill(180, 100, 100)  # 建筑：红棕色
        py5.stroke(120, 60, 60)
    else:
        py5.fill(220, 220, 210)  # block：浅灰色
        py5.stroke(80, 80, 80)

    py5.begin_shape()
    for x, y in coords:
        py5.vertex(x, y)
    py5.end_shape(py5.CLOSE)


def draw_current_block():
    global current_index
    if current_index < 0 or current_index >= len(blocks):
        return

    block = blocks[current_index]
    block_id = block.block_id
    block_geom = block.geom

    buildings_geoms = buildings_data[block_id]

    py5.background(240)

    W, H = py5.width, py5.height
    minx, miny, maxx, maxy = block_geom.bounds
    cw = maxx - minx
    ch = maxy - miny
    margin = 0.15
    scale = min((W * (1 - margin)) / cw, (H * (1 - margin)) / ch)
    cx = (minx + maxx) / 2
    cy = (miny + maxy) / 2

    py5.push_matrix()
    py5.translate(W / 2, H / 2)
    py5.scale(scale, -scale)
    py5.translate(-cx, -cy)

    # 先画block轮廓
    py5.stroke_weight(1 / scale)
    for p in all_polygons(block_geom):
        draw_polygon(p, is_building=False)

    # 再画建筑footprint
    py5.stroke_weight(0.5 / scale)
    for geom in buildings_geoms:
        for p in all_polygons(geom):
            draw_polygon(p, is_building=True)

    py5.pop_matrix()

    # 显示当前block信息
    py5.fill(0)
    py5.text_size(16)
    py5.text(f"Block: {block_id} ({current_index + 1}/{len(blocks)})", 10, 20)


def setup():
    py5.size(900, 900)
    draw_current_block()


def draw():
    pass  # 静态绘制


def key_pressed():
    global current_index
    if py5.key == '1':
        current_index = max(0, current_index - 1)
        draw_current_block()
    elif py5.key == '2':
        current_index = min(len(blocks) - 1, current_index + 1)
        draw_current_block()


py5.run_sketch()


随机选取了 50 个block


### 2 处理lod2的CityGML数据

#### 2.1 测试：任选一个lod2的xml并转换为json，然后解析数据做检查

In [3]:
citygml_tools_path = r"E:\0_mylib\citygml-tools-2.4.1\citygml-tools.bat"

utils_z.run_cmd(f'"{citygml_tools_path}" --version')

citygml-tools version 2.4.1
(C) 2018-2026 Claus Nagel <claus.nagel@gmail.com>




CompletedProcess(args='"E:\\0_mylib\\citygml-tools-2.4.1\\citygml-tools.bat" --version', returncode=0, stdout='citygml-tools version 2.4.1\n(C) 2018-2026 Claus Nagel <claus.nagel@gmail.com>\n\n', stderr='')

In [4]:
test_xml_path = "E:\\2_data\\building_3d_opensource\\hamburg\\LoD2-DE_HH_2025-03-14\\LoD2_32_557_5932_1_HH.xml"

utils_z.run_cmd(f'"{citygml_tools_path}" to-cityjson {test_xml_path}')

[11:40:02 INFO] Starting citygml-tools.
[11:40:02 INFO] Executing 'to-cityjson' command.
[11:40:02 INFO] [1|1] Processing file E:\2_data\building_3d_opensource\hamburg\LoD2-DE_HH_2025-03-14\LoD2_32_557_5932_1_HH.xml.
[11:40:03 INFO] Writing output to file E:\2_data\building_3d_opensource\hamburg\LoD2-DE_HH_2025-03-14\LoD2_32_557_5932_1_HH.json.
[11:40:03 INFO] Total execution time: 01 s.
[11:40:03 INFO] citygml-tools successfully completed.



CompletedProcess(args='"E:\\0_mylib\\citygml-tools-2.4.1\\citygml-tools.bat" to-cityjson E:\\2_data\\building_3d_opensource\\hamburg\\LoD2-DE_HH_2025-03-14\\LoD2_32_557_5932_1_HH.xml', returncode=0, stdout="[11:40:02 INFO] Starting citygml-tools.\n[11:40:02 INFO] Executing 'to-cityjson' command.\n[11:40:02 INFO] [1|1] Processing file E:\\2_data\\building_3d_opensource\\hamburg\\LoD2-DE_HH_2025-03-14\\LoD2_32_557_5932_1_HH.xml.\n[11:40:03 INFO] Writing output to file E:\\2_data\\building_3d_opensource\\hamburg\\LoD2-DE_HH_2025-03-14\\LoD2_32_557_5932_1_HH.json.\n[11:40:03 INFO] Total execution time: 01 s.\n[11:40:03 INFO] citygml-tools successfully completed.\n", stderr='')

In [5]:
import json

test_json_path = test_xml_path.replace('.xml', '.json')
with open(test_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 查看顶层结构
print(data.keys())

# 查看第一栋建筑的属性
first_building_id = list(data["CityObjects"].keys())[0]
first_building = data["CityObjects"][first_building_id]
print(f"\n建筑ID：{first_building_id}")
print(f"类型：{first_building['type']}")
print(f"属性：{json.dumps(first_building.get('attributes', {}), indent=2, ensure_ascii=False)}")
print(f"几何类型：{[g['type'] for g in first_building.get('geometry', [])]}")

dict_keys(['type', 'version', 'CityObjects', 'transform', 'vertices', 'metadata'])

建筑ID：DEHHALKAJ0000nVM
类型：Building
属性：{
  "creationDate": "2023-02-23T00:00:00+08:00",
  "DatenquelleLage": "1000",
  "DatenquelleBodenhoehe": "1100",
  "Gemeindeschluessel": "02100141",
  "DatenquelleDachhoehe": "1000",
  "DatenquelleGeschossanzahl": "1000",
  "Geometrietyp2DReferenz": "3000",
  "Grundrissaktualitaet": "2025-01-27",
  "measuredHeight": 2.297,
  "function": "31001_2460",
  "roofType": "1000",
  "storeysAboveGround": 1
}
几何类型：['Solid']


In [6]:
# 查看LOD2的几何边界详细结构
first_geom = first_building["geometry"][0]
boundaries = first_geom["boundaries"]
semantics = first_geom.get("semantics", {})

print(f"semantics keys: {semantics.keys()}")
print(f"surfaces: {json.dumps(semantics.get('surfaces', []), indent=2)}")
print(f"values: {semantics.get('values', [])}")

semantics keys: dict_keys(['surfaces', 'values'])
surfaces: [
  {
    "type": "RoofSurface",
    "id": "DEHH_30323b6e-5063-4083-883a-1cbae16a9c77_2"
  },
  {
    "type": "GroundSurface",
    "id": "DEHH_9412edc4-eaa9-487a-8d63-74f41a226b8c_2"
  },
  {
    "type": "WallSurface",
    "id": "DEHH_407ba734-a030-411d-9759-833481c0fe2c_2"
  },
  {
    "type": "WallSurface",
    "id": "DEHH_94c46c85-5e4c-4464-82e4-56442979e7f4_2"
  },
  {
    "type": "WallSurface",
    "id": "DEHH_3a29fc3f-a942-4a04-8727-971447d615c2_2"
  },
  {
    "type": "WallSurface",
    "id": "DEHH_bee6e924-cc32-48d2-b198-f0f750fe20d5_2"
  }
]
values: [[0, 1, 2, 3, 4, 5]]


In [8]:
with open(test_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

first_id = list(data["CityObjects"].keys())[0]
first_obj = data["CityObjects"][first_id]
geom_entry = next((g for g in first_obj.get("geometry", []) if str(g.get("lod")) == "2"), None)

semantics = geom_entry.get("semantics", {})
print(f"surfaces: {json.dumps(semantics.get('surfaces', []), indent=2)}")
print(f"values原始结构: {semantics.get('values')}")
print(f"values类型: {type(semantics.get('values'))}")
print(f"values长度: {len(semantics.get('values', []))}")

surfaces: [
  {
    "type": "RoofSurface",
    "id": "DEHH_30323b6e-5063-4083-883a-1cbae16a9c77_2"
  },
  {
    "type": "GroundSurface",
    "id": "DEHH_9412edc4-eaa9-487a-8d63-74f41a226b8c_2"
  },
  {
    "type": "WallSurface",
    "id": "DEHH_407ba734-a030-411d-9759-833481c0fe2c_2"
  },
  {
    "type": "WallSurface",
    "id": "DEHH_94c46c85-5e4c-4464-82e4-56442979e7f4_2"
  },
  {
    "type": "WallSurface",
    "id": "DEHH_3a29fc3f-a942-4a04-8727-971447d615c2_2"
  },
  {
    "type": "WallSurface",
    "id": "DEHH_bee6e924-cc32-48d2-b198-f0f750fe20d5_2"
  }
]
values原始结构: [[0, 1, 2, 3, 4, 5]]
values类型: <class 'list'>
values长度: 1


In [10]:
test_buildings = cjpar.parse_cityjson_lod2(test_json_path)
print(f"解析出建筑数：{len(test_buildings)}")
b = test_buildings[0]
print(f"ID：{b['building_id']}")
print(f"高度：{b['height']}")
print(f"2D footprint：{b['geom_2d']}")
print(f"RoofSurface数量：{len(b['surfaces']['RoofSurface'])}")
print(f"WallSurface数量：{len(b['surfaces']['WallSurface'])}")
print(f"GroundSurface数量：{len(b['surfaces']['GroundSurface'])}")

解析出建筑数：420
ID：DEHHALKAJ0000nVM
高度：2.297
2D footprint：POLYGON ((557616.316 5932088.971, 557616.334 5932094.35, 557624.841 5932094.319, 557624.8230000001 5932088.939, 557616.316 5932088.971))
RoofSurface数量：1
WallSurface数量：4
GroundSurface数量：1


#### 2.2 创建LOD2建筑表并批量转换、导入数据

In [13]:
# LOD2建筑主表
utils_z.run_sql("""
    CREATE TABLE IF NOT EXISTS hamburg_buildings_lod2 (
        building_id     VARCHAR PRIMARY KEY,
        block_id        VARCHAR,
        geom_2d         GEOMETRY(Polygon, 25832),
        height          FLOAT,
        floor_count     INTEGER,
        function        VARCHAR,
        roof_type       VARCHAR,
        year_built      INTEGER
    );
""", conn=conn)

utils_z.run_sql("""
    CREATE INDEX IF NOT EXISTS buildings_lod2_geom_idx
    ON hamburg_buildings_lod2 USING GIST (geom_2d);
""", conn=conn)

# LOD2 surfaces子表
utils_z.run_sql("""
    CREATE TABLE IF NOT EXISTS hamburg_building_surfaces_lod2 (
        surface_id      SERIAL PRIMARY KEY,
        building_id     VARCHAR REFERENCES hamburg_buildings_lod2(building_id),
        surface_type    VARCHAR,
        geom_3d         GEOMETRY(PolygonZ, 25832)
    );
""", conn=conn)

utils_z.run_sql("""
    CREATE INDEX IF NOT EXISTS surfaces_lod2_building_idx
    ON hamburg_building_surfaces_lod2 (building_id);
""", conn=conn)

print("LOD2表创建完成")

LOD2表创建完成


In [14]:
# 先批量转换XML到CityJSON
lod2_xml_dir = r"E:\2_data\building_3d_opensource\hamburg\LoD2-DE_HH_2025-03-14"
lod2_json_dir = r"E:\2_data\building_3d_opensource\hamburg\lod2_json"
os.makedirs(lod2_json_dir, exist_ok=True)

xml_files = [f for f in os.listdir(lod2_xml_dir) if f.endswith(".xml")]
print(f"共{len(xml_files)}个文件待转换")

errors = []
for i, filename in enumerate(xml_files):
    input_path = os.path.join(lod2_xml_dir, filename)
    cmd = f'"{citygml_tools_path}" to-cityjson --output="{lod2_json_dir}" "{input_path}"'
    try:
        utils_z.run_cmd(cmd, False)
        if (i + 1) % 50 == 0:
            print(f"转换进度：{i + 1}/{len(xml_files)}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"转换错误：{filename} -> {e}")

print(f"转换完成，失败{len(errors)}个")

共782个文件待转换
转换进度：50/782
转换进度：100/782
转换进度：150/782
转换进度：200/782
转换进度：250/782
转换进度：300/782
转换进度：350/782
转换进度：400/782
转换进度：450/782
转换进度：500/782
转换进度：550/782
转换进度：600/782
转换进度：650/782
转换进度：700/782
转换进度：750/782
转换完成，失败0个


In [15]:
json_files = [f for f in os.listdir(lod2_json_dir) if f.endswith(".json")]
print(f"共{len(json_files)}个文件待入库")

total = 0
errors = []

for i, filename in enumerate(json_files):
    filepath = os.path.join(lod2_json_dir, filename)
    try:
        buildings = cjpar.parse_cityjson_lod2(filepath)
        count = cjpar.insert_buildings_lod2(buildings, conn,
                                       lod2_table="hamburg_buildings_lod2",
                                       surface_table="hamburg_building_surfaces_lod2")
        total += count
        if (i + 1) % 50 == 0:
            print(f"入库进度：{i+1}/{len(json_files)}，已入库建筑：{total}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"错误：{filename} -> {e}")

print(f"\n完成！共入库建筑：{total}")
print(f"失败文件数：{len(errors)}")

共782个文件待入库
入库进度：50/782，已入库建筑：15480
入库进度：100/782，已入库建筑：39286
入库进度：150/782，已入库建筑：69711
入库进度：200/782，已入库建筑：90508
入库进度：250/782，已入库建筑：118378
入库进度：300/782，已入库建筑：154574
入库进度：350/782，已入库建筑：181148
入库进度：400/782，已入库建筑：209042
入库进度：450/782，已入库建筑：229861
入库进度：500/782，已入库建筑：256772
入库进度：550/782，已入库建筑：294236
入库进度：600/782，已入库建筑：317748
入库进度：650/782，已入库建筑：344815
入库进度：700/782，已入库建筑：365178
入库进度：750/782，已入库建筑：384626

完成！共入库建筑：388267
失败文件数：0


#### 2.3 更新block_id字段并空间叠合

In [16]:
# 添加block_id字段
utils_z.run_sql("""
    ALTER TABLE hamburg_buildings_lod2
    ADD COLUMN IF NOT EXISTS block_id VARCHAR;
""", conn=conn)

# 建立空间索引
utils_z.run_sql("""
    CREATE INDEX IF NOT EXISTS buildings_lod2_geom_idx
    ON hamburg_buildings_lod2 USING GIST (geom_2d);
""", conn=conn)

print("开始空间叠合...")

开始空间叠合...


In [17]:
utils_z.run_sql("""
    UPDATE hamburg_buildings_lod2 b
    SET block_id = (
        SELECT bl.block_id
        FROM hamburg_blocks bl
        WHERE ST_Within(ST_Centroid(b.geom_2d), bl.geom)
        LIMIT 1
    );
""", conn=conn)
print("空间叠合完成")

空间叠合完成


In [18]:
# 检查匹配情况
result = utils_z.run_sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(block_id) AS matched,
        COUNT(*) FILTER (WHERE block_id IS NULL) AS unmatched
    FROM hamburg_buildings_lod2;
""", conn=conn, fetch=True)

print(f"总建筑数：{result[0][0]}")
print(f"成功匹配block：{result[0][1]}")
print(f"未匹配block：{result[0][2]}")

总建筑数：388267
成功匹配block：357535
未匹配block：30732


#### 2.4 绘制检查，轴测三维视角

In [4]:
import geopandas as gpd
import numpy as np
import py5

In [ ]:
# 查询30个有LOD2建筑的block
sql_blocks = """
    SELECT bl.block_id, bl.geom
    FROM hamburg_blocks bl
    WHERE EXISTS (
        SELECT 1 FROM hamburg_buildings_lod2 b
        WHERE b.block_id = bl.block_id
    )
    ORDER BY RANDOM()
    LIMIT 30;
"""
gdf_blocks = gpd.read_postgis(sql_blocks, conn, geom_col="geom", crs="EPSG:25832")
block_ids = list(gdf_blocks["block_id"])
print(f"已选取{len(block_ids)}个block")


def fetch_surfaces(block_id):
    """查询某个block内所有建筑的LOD2 surfaces"""
    sql = f"""
        SELECT s.surface_type, ST_AsText(s.geom_3d) AS geom_wkt
        FROM hamburg_building_surfaces_lod2 s
        JOIN hamburg_buildings_lod2 b ON s.building_id = b.building_id
        WHERE b.block_id = '{block_id}';
    """
    return utils_z.run_sql(sql, conn=conn, fetch=True)

def fetch_block_geom(block_id):
    """查询block轮廓顶点"""
    sql = f"""
        SELECT ST_AsText(geom) FROM hamburg_blocks WHERE block_id = '{block_id}';
    """
    result = utils_z.run_sql(sql, conn=conn, fetch=True)
    from shapely.wkt import loads
    return loads(result[0][0])

def parse_wkt_polygon_3d(wkt):
    """从WKT解析3D顶点"""
    from shapely.wkt import loads
    poly = loads(wkt)
    return [coord for coord in poly.exterior.coords]

def get_min_z(surfaces):
    """找到所有surfaces中最低的Z值作为地面基准"""
    min_z = float('inf')
    for stype, wkt in surfaces:
        try:
            coords = parse_wkt_polygon_3d(wkt)
            for coord in coords:
                if len(coord) > 2:
                    min_z = min(min_z, coord[2])
        except:
            continue
    return min_z if min_z != float('inf') else 0

# 全局状态
current_idx = [0]
cached_data = {}

def get_data(idx):
    """获取并缓存当前block数据"""
    block_id = str(block_ids[idx])  # 强制转换为字符串
    if block_id not in cached_data:
        # print(f"加载 {block_id}...")
        surfaces = fetch_surfaces(block_id)
        block_geom = fetch_block_geom(block_id)
        cached_data[block_id] = (block_geom, surfaces)
    return cached_data[block_id]


def to_screen_3d(x, y, z, base_z, cx, cy):
    """直接返回3D坐标，让py5处理投影"""
    return (x - cx, y - cy, z - base_z)

def draw_scene(idx):
    py5.background(240)
    block_id = str(block_ids[idx])
    block_geom, surfaces = get_data(idx)
    base_z = get_min_z(surfaces)

    # 计算中心点
    minx, miny, maxx, maxy = block_geom.bounds
    cx = (minx + maxx) / 2
    cy = (miny + maxy) / 2
    extent = max(maxx - minx, maxy - miny)
    scale = py5.width * 0.6 / extent

    py5.push_matrix()
    py5.translate(py5.width/2, py5.height/2, 0)
    py5.scale(scale, -scale, scale)  # y轴翻转
    py5.rotate_x(py5.radians(-30))   # 俯视角
    py5.rotate_z(py5.radians(-45))  # 旋转方向

    # 绘制block轮廓
    py5.stroke(80, 80, 80)
    py5.fill(220, 220, 210)
    py5.stroke_weight(1/scale)
    coords = list(block_geom.exterior.coords)
    py5.begin_shape()
    for coord in coords:
        py5.vertex(coord[0]-cx, coord[1]-cy, 0)
    py5.end_shape(py5.CLOSE)

    # 绘制建筑surfaces
    color_map = {
        "RoofSurface":   (180, 80,  80),
        "WallSurface":   (200, 160, 120),
        "GroundSurface": (150, 150, 140),
    }
    py5.stroke_weight(0.3/scale)
    for stype, wkt in surfaces:
        try:
            coords_3d = parse_wkt_polygon_3d(wkt)
            r, g, b = color_map.get(stype, (160, 160, 160))
            py5.fill(r, g, b)
            py5.stroke(max(r-40,0), max(g-40,0), max(b-40,0))
            py5.begin_shape()
            for coord in coords_3d:
                z = coord[2] if len(coord) > 2 else base_z
                py5.vertex(coord[0]-cx, coord[1]-cy, z-base_z)
            py5.end_shape(py5.CLOSE)
        except:
            continue

    py5.pop_matrix()

    # 文字标注（回到2D模式）
    py5.fill(0)
    py5.text_size(14)
    py5.text(f"Block: {block_id}  [{idx+1}/30]", 20, 30)
    py5.text("1=prev  2=next", 20, 55)

def setup():
    py5.size(1000, 800, py5.P3D)
    py5.text_font(py5.create_font("Arial", 14))
    draw_scene(current_idx[0])

def draw():
    pass

def key_pressed():
    if py5.key == '1':
        current_idx[0] = (current_idx[0] - 1) % len(block_ids)
        draw_scene(current_idx[0])
    elif py5.key == '2':
        current_idx[0] = (current_idx[0] + 1) % len(block_ids)
        draw_scene(current_idx[0])

py5.run_sketch()

已选取30个block


加载 DE_HH_001841...
加载 DE_HH_006339...
加载 DE_HH_001255...
加载 DE_HH_002032...
加载 DE_HH_005385...
加载 DE_HH_003389...
加载 DE_HH_003767...
加载 DE_HH_001914...
加载 DE_HH_004263...
加载 DE_HH_001851...
加载 DE_HH_005134...
加载 DE_HH_002622...
加载 DE_HH_006465...
加载 DE_HH_003393...
加载 DE_HH_000871...
加载 DE_HH_002371...
加载 DE_HH_003810...
加载 DE_HH_006141...
加载 DE_HH_004318...
加载 DE_HH_000537...
加载 DE_HH_002675...
加载 DE_HH_000763...
加载 DE_HH_001184...
加载 DE_HH_005374...
加载 DE_HH_003498...
加载 DE_HH_001935...
加载 DE_HH_005777...
加载 DE_HH_002416...
加载 DE_HH_000827...
加载 DE_HH_004693...


In [33]:
# 检查某个block的建筑Z值范围
block_id = str(block_ids[0])
surfaces = fetch_surfaces(block_id)

z_values = []
for stype, wkt in surfaces:
    try:
        coords = parse_wkt_polygon_3d(wkt)
        for coord in coords:
            if len(coord) > 2:
                z_values.append(coord[2])
    except:
        continue

print(f"Z最小值：{min(z_values):.3f}")
print(f"Z最大值：{max(z_values):.3f}")
print(f"Z范围：{max(z_values) - min(z_values):.3f}")

# 同时看block轮廓的坐标
block_geom = fetch_block_geom(block_id)
print(f"block轮廓坐标示例：{list(block_geom.exterior.coords)[:2]}")

Z最小值：4.580
Z最大值：32.759
Z范围：28.179
block轮廓坐标示例：[(580760.3018368954, 5927557.591860396), (580647.6929736413, 5927529.931834438)]
